# Atividade 2 - Mineração de texto

##Importações

* Instalações Colab

* Importações Bibliotecas

In [1]:
!pip install -q unidecode nltk spacy
!python -m spacy download pt_core_news_sm -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 76.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import re, string, itertools
import nltk, spacy, unidecode
from nltk.corpus import stopwords
from nltk.tokenize import wordpunct_tokenize, sent_tokenize

#recursos NLTK
nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)

#modelo spaCy
nlp = spacy.load("pt_core_news_sm")

## Frases Testes

In [3]:
texto_base = ("Olá! Este é um teste número 123. ""Vamos verificar os espaços, pontuações e números!")

## Exercicio 1

* Limpeza

In [4]:
def limpa(txt: str) -> str:
    txt = txt.lower()
    txt = re.sub(r"\d+", "", txt)
    txt = txt.translate(str.maketrans("", "", string.punctuation))
    txt = re.sub(r"\s+", " ", txt).strip()
    return txt

texto_limpo = limpa(texto_base)
print(texto_limpo)

olá este é um teste número vamos verificar os espaços pontuações e números


## Exercicio 2

In [5]:
#Tokenização
def tokeniza_pal(txt):
    return wordpunct_tokenize(txt)

def tokeniza_sent(txt):
    return [sent.text.strip() for sent in nlp(txt).sents]

tokens_pal = tokeniza_pal(texto_limpo)
tokens_sent = tokeniza_sent(texto_base)
print("Tokenização palavras:", tokens_pal)
print("Tokenização sentenças:", tokens_sent, "\n")

Tokenização palavras: ['olá', 'este', 'é', 'um', 'teste', 'número', 'vamos', 'verificar', 'os', 'espaços', 'pontuações', 'e', 'números']
Tokenização sentenças: ['Olá!', 'Este é um teste número 123.', 'Vamos verificar os espaços, pontuações e números!'] 



## Exercicio 3

* Remoção StopWords

In [6]:
def remove_sw(tok_list):
    sw = set(stopwords.words("portuguese"))
    return [t for t in tok_list if t not in sw]

tokens_sem_sw = remove_sw(tokens_pal)
print("Sem stopwords:", tokens_sem_sw, "\n")

Sem stopwords: ['olá', 'teste', 'número', 'vamos', 'verificar', 'espaços', 'pontuações', 'números'] 



## Exercicio 4

### N-Gramas

* N-gramas é uma uma sequencia unitaria de tokens, somente palavras unicas sem continuação e sem antecedentes, o aumento de gramas aumenta o nivel de contexto, é dito que para prever as proximas palavras podemos avaliar o numero limitado de itens anteriores com isso trazendo maior contexto.

In [7]:
def gera_ngrams(tokens, n):
    return list(zip(*(tokens[i:] for i in range(n))))

unigrams = tokens_sem_sw
bigrams  = gera_ngrams(tokens_sem_sw, 2)
trigrams = gera_ngrams(tokens_sem_sw, 3)
print("Unigramas:", unigrams)
print("Bigramas :", bigrams)
print("Trigramas:", trigrams, "\n")

Unigramas: ['olá', 'teste', 'número', 'vamos', 'verificar', 'espaços', 'pontuações', 'números']
Bigramas : [('olá', 'teste'), ('teste', 'número'), ('número', 'vamos'), ('vamos', 'verificar'), ('verificar', 'espaços'), ('espaços', 'pontuações'), ('pontuações', 'números')]
Trigramas: [('olá', 'teste', 'número'), ('teste', 'número', 'vamos'), ('número', 'vamos', 'verificar'), ('vamos', 'verificar', 'espaços'), ('verificar', 'espaços', 'pontuações'), ('espaços', 'pontuações', 'números')] 



## Exercicio 4

* POS-tagging + NER

In [8]:
def spacy_infos(txt):
    d = nlp(txt)
    verbos = [t.text for t in d if t.pos_ == "VERB"]
    substantivos = [t.text for t in d if t.pos_ == "NOUN"]
    ents = [(e.text, e.label_) for e in d.ents]
    return verbos, substantivos, ents

verbos, subs, ents = spacy_infos(texto_limpo)
print("Verbos:", verbos)
print("Substantivos:", subs)
print("Entidades:", ents, "\n")


Verbos: ['olá', 'verificar']
Substantivos: ['teste', 'número', 'espaços', 'pontuações', 'números']
Entidades: [('olá', 'MISC')] 



## Exercicio 5

### Importancia dos filtros

* funcionam como regras gramaticais linguisticas, onde tentamos entender caracteristicas no texto por meio de quatro filtros sendo estes NOUN, ADJ, VERB, ADV, se tentarmos entender naturamente seria o noun o foco principal do texto ou aquilo que gera o sentido da frase, adj seriam as caracteristicas de algo, Verb seria a ação dentro do texto ou estado, e adv seria tempo ou lugar em quesito de estado, com isso extraimos pontos importantes dentro de pesquisas simples com poucos passos garantindo nivel de certo nivel de exatidão nos resuldados obtidos.

*  Filtro gramatical

In [9]:
def filtra_classes(txt):
    d = nlp(txt)
    keep = {"NOUN", "ADJ", "VERB", "ADV"}
    return [t.text for t in d if t.pos_ in keep]

tokens_filtr = filtra_classes(texto_limpo)
print("Filtrado:", " ".join(tokens_filtr))

Filtrado: olá teste número verificar espaços pontuações números
